# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/helnagar123/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 0. Set Up

In [1]:
!pip install -q datasets duckdb scikit-learn pandas numpy matplotlib

In [2]:
import duckdb
import pandas as pd
import numpy as np

from datasets import load_dataset
from huggingface_hub import login
from google.colab import userdata

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)

from sklearn.ensemble import RandomForestClassifier

In [3]:
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

In [4]:
content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train"
)

performance = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train"
)

client = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train"
)

con = duckdb.connect()

con.register("dim_content", content.data.table)
con.register("daily_perf", performance.data.table)
con.register("dim_client", client.data.table)

print("Tables loaded successfully!")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

...

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/104 [00:00<?, ? examples/s]

Tables loaded successfully!


In [5]:
con.sql("""
CREATE OR REPLACE VIEW refresh_features AS
SELECT
    p.report_date,

    c.client_hash_id,
    c.content_hash_id,

    c.content_created_date,
    c.content_updated_date,
    c.last_optimized_date,

    c.search_volume,
    c.backlinks,
    c.word_count,

    p.gsc_impressions,
    p.gsc_clicks,
    p.gsc_avg_position,

    CASE
        WHEN p.gsc_impressions > 0
        THEN (100.0 * p.gsc_clicks / p.gsc_impressions)
        ELSE NULL
    END AS ctr,

    DATE_DIFF('day', p.report_date, c.content_updated_date) AS content_age_days

FROM daily_perf p

JOIN dim_content c
USING(content_hash_id)

WHERE
    p.gsc_data_available = TRUE
    AND c.is_published = TRUE
    AND c.is_deleted = FALSE
""")

print("View created.")

View created.


In [6]:
feature_df = con.sql("""

SELECT

    client_hash_id,
    content_hash_id,

    content_age_days,

    gsc_impressions AS impressions,
    gsc_clicks AS clicks,

    ctr,

    gsc_avg_position AS position,

    search_volume,
    backlinks,
    word_count

FROM refresh_features

USING SAMPLE 1000000 ROWS

""").df()

feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,content_age_days,impressions,clicks,ctr,position,search_volume,backlinks,word_count
0,client_73cda7b4e4f265ea,content_76de2fc8e679b574,103,1,0,0.0,0.000000,20,<NA>,<NA>
1,client_73cda7b4e4f265ea,content_7d0b10c6c1c00e44,51,147,0,0.0,2.190476,0,<NA>,2796
2,client_3197e6291363b4db,content_6ddccd0500b60eac,22,32,0,0.0,12.093750,6600,<NA>,3591
3,client_3197e6291363b4db,content_5807e23ec24ff22d,-90,7,0,0.0,83.142857,20,<NA>,3341
4,client_08a6a72ff48e62c0,content_01f5a4cbcfc34e24,-22,1,0,0.0,57.000000,10,0,<NA>


In [7]:
model_df = feature_df.copy()

model_df["word_count"] = model_df["word_count"].fillna(0)
model_df["search_volume"] = model_df["search_volume"].fillna(0)
model_df["backlinks"] = model_df["backlinks"].fillna(0)

model_df = model_df.dropna()

print(model_df.shape)

(1000000, 10)


In [8]:
model_df = model_df.sort_values(
    "ctr",
    ascending=False
).reset_index(drop=True)

model_df["target"] = 0

top_n = int(len(model_df) * 0.20)

model_df.loc[:top_n-1, "target"] = 1

model_df["target"].value_counts()

,count
target,
0,800000
1,200000


In [9]:
feature_columns = [
    "content_age_days",
    "impressions",
    "clicks",
    "position",
    "search_volume",
    "backlinks",
    "word_count",
]

X = model_df[feature_columns]

y = model_df["target"]

groups = model_df["client_hash_id"]

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [10]:
def precision_at_k(y_true, scores, k=20):

    results = pd.DataFrame({
        "target": y_true.values,
        "score": scores
    })

    top_k = results.sort_values(
        "score",
        ascending=False
    ).head(k)

    return top_k["target"].mean()

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    random_state=42
)

rf.fit(X_train, y_train)

random_scores = rf.predict_proba(X_test)[:, 1]

random_precision = precision_at_k(
    y_test,
    random_scores,
    k=100
)

print(f"Random Split Precision@100 = {random_precision:.3f}")

Random Split Precision@100 = 1.000


In [18]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

rf = RandomForestClassifier(
    random_state=42
)

rf.fit(X_train, y_train)

group_scores = rf.predict_proba(X_test)[:, 1]

group_precision = precision_at_k(
    y_test,
    group_scores,
    k=100
)

print(f"Grouped Split Precision@100 = {group_precision:.3f}")

Grouped Split Precision@100 = 1.000


In [19]:
comparison = pd.DataFrame({
    "Validation Strategy": [
        "Random Split",
        "Grouped Split"
    ],
    "Precision@20": [
        random_precision,
        group_precision
    ]
})

comparison

,Validation Strategy,Precision@20
0,Random Split,1.0
1,Grouped Split,1.0


# 3. Leakage Audit

This section reviews the final feature set for potential leakage.

The goal is to identify whether any feature contains information that would not realistically be available at prediction time or is too closely related to the target definition.

In [20]:
leakage_audit = pd.DataFrame({
    "Feature": feature_columns,
    "Potential Leakage": [
        "No",
        "Yes",
        "Yes",
        "No",
        "No",
        "No",
        "No"
    ],
    "Reason": [
        "Available before making refresh decisions.",
        "Used to calculate the target (CTR depends on impressions and clicks).",
        "Used to calculate the target (CTR depends on impressions and clicks).",
        "Historical search performance.",
        "Keyword-level metadata.",
        "Historical backlink count.",
        "Static content metadata."
    ]
})

leakage_audit

,Feature,Potential Leakage,Reason
0,content_age_days,No,Available before making refresh decisions.
1,impressions,Yes,Used to calculate the target (CTR depends on i...
2,clicks,Yes,Used to calculate the target (CTR depends on i...
3,position,No,Historical search performance.
4,search_volume,No,Keyword-level metadata.
5,backlinks,No,Historical backlink count.
6,word_count,No,Static content metadata.


### Findings

The audit shows that **clicks** and **impressions** are closely related to the target because the target was created from the top 20% of CTR values, and CTR is calculated from these two variables.

This is not direct feature leakage because CTR itself is not used as a feature. However, these predictors provide information that is highly correlated with the target, making the prediction task much easier.

For a production model, the target should ideally be based on future outcomes rather than a metric derived from the same historical inputs.

# 4. Claim Rewrite

### Original claim

> The model identifies the pages that should be refreshed with high accuracy.

### Revised claim

> In this experiment, the model achieved a high Precision@100 on the available dataset. This result is an observed measurement under the current validation setup and should be interpreted as decision-support rather than proof that the model will generalize to all future content or clients.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.